<a href="https://colab.research.google.com/github/realshubhamraut/CDAC-DBDA-coursework/blob/main/06.big-data-technologies/assignments/assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import os
def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
  !java -version
install_java()

openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)


# PySpark Airlines Dataset Assignment

Dataset: `/data/airlines.csv`

## Sections:
1. **Questions 1-9**: DataFrame API
2. **Questions 10-19**: Spark SQL
3. **Questions 20-39**: Data Analysis

## Initialize Spark Session

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, max, desc
import os

# Path where CSV will be stored inside Colab
file_path = "/content/airlines.csv"

# Raw GitHub file link
url = "https://raw.githubusercontent.com/realshubhamraut/CDAC-DBDA-coursework/main/06.big-data-technologies/data/airlines.csv"

# Download the file only if not present in current runtime
if not os.path.exists(file_path):
    !wget -q {url} -O {file_path}
    print("Dataset downloaded")
else:
    print("Dataset already available in runtime")

# Create Spark Session
spark = SparkSession.builder \
    .appName("Airlines Dataset Analysis") \
    .getOrCreate()

# Load dataset from local runtime
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display data & schema
df.show()
df.printSchema()
df.show(5)

✅ Dataset downloaded
+----+-------+----------------+------------+
|Year|Quarter|Avg_rev_per_seat|booked_seats|
+----+-------+----------------+------------+
|1995|      1|           296.9|       46561|
|1995|      2|           296.8|       37443|
|1995|      3|          287.51|       34128|
|1995|      4|          287.78|       30388|
|1996|      1|          283.97|       47808|
|1996|      2|          275.78|       43020|
|1996|      3|          269.49|       38952|
|1996|      4|          278.33|       37443|
|1997|      1|           283.4|       35067|
|1997|      2|          289.44|       46565|
|1997|      3|          282.27|       38886|
|1997|      4|          293.51|       37454|
|1998|      1|          304.74|       31315|
|1998|      2|          300.97|       30852|
|1998|      3|          315.25|       38118|
|1998|      4|          316.18|       35393|
|1999|      1|          331.74|       47453|
|1999|      2|          329.34|       38243|
|1999|      3|          317.22|   

**Note:** All revenue calculations are displayed in millions for better readability. For example, `46.36` means $46.36 million.

---
# Part 1: DataFrame API (Questions 1-9)

### Q1. Calculate the average revenue per seat for each year and quarter

In [5]:
# Q1: Average revenue per seat for each year and quarter
avg_revenue_per_year_quarter = df.groupBy("Year", "Quarter") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy("Year", "Quarter")

avg_revenue_per_year_quarter.show()

+----+-------+--------------------+
|Year|Quarter|Avg_Revenue_Per_Seat|
+----+-------+--------------------+
|1995|      1|               296.9|
|1995|      2|               296.8|
|1995|      3|              287.51|
|1995|      4|              287.78|
|1996|      1|              283.97|
|1996|      2|              275.78|
|1996|      3|              269.49|
|1996|      4|              278.33|
|1997|      1|               283.4|
|1997|      2|              289.44|
|1997|      3|              282.27|
|1997|      4|              293.51|
|1998|      1|              304.74|
|1998|      2|              300.97|
|1998|      3|              315.25|
|1998|      4|              316.18|
|1999|      1|              331.74|
|1999|      2|              329.34|
|1999|      3|              317.22|
|1999|      4|              317.93|
+----+-------+--------------------+
only showing top 20 rows



#### RDD API Solution

In [ ]:
# Q1 RDD: Average revenue per seat for each year and quarter
rdd = df.rdd

# Map to ((Year, Quarter), Avg_rev_per_seat)
year_quarter_revenue = rdd.map(lambda row: ((row.Year, row.Quarter), row.Avg_rev_per_seat))

# Calculate average using aggregateByKey
def seq_op(acc, value):
    return (acc[0] + value, acc[1] + 1)

def comb_op(acc1, acc2):
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

avg_rdd = year_quarter_revenue.aggregateByKey((0.0, 0), seq_op, comb_op) \
    .mapValues(lambda x: x[0] / x[1]) \
    .sortByKey()

print("Q1 RDD Result:")
for key, value in avg_rdd.collect():
    print(f"Year: {key[0]}, Quarter: {key[1]}, Avg Revenue Per Seat: {value:.2f}")

### Q2. Find the year and quarter with the highest average revenue per seat

In [6]:
# Q2: Year and quarter with highest average revenue per seat
highest_avg_revenue = df.orderBy(desc("Avg_rev_per_seat")).limit(1)
highest_avg_revenue.show()

# Alternative: Using groupBy
highest_avg_revenue_alt = df.groupBy("Year", "Quarter") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy(desc("Avg_Revenue_Per_Seat")) \
    .limit(1)
highest_avg_revenue_alt.show()

+----+-------+----------------+------------+
|Year|Quarter|Avg_rev_per_seat|booked_seats|
+----+-------+----------------+------------+
|2014|      3|          396.37|       40257|
+----+-------+----------------+------------+

+----+-------+--------------------+
|Year|Quarter|Avg_Revenue_Per_Seat|
+----+-------+--------------------+
|2014|      3|              396.37|
+----+-------+--------------------+



#### RDD API Solution

In [ ]:
# Q2 RDD: Year and quarter with highest average revenue per seat
# Find max by sorting by Avg_rev_per_seat in descending order
highest_avg_rdd = rdd.map(lambda row: (row.Year, row.Quarter, row.Avg_rev_per_seat)) \
    .sortBy(lambda x: x[2], ascending=False) \
    .take(1)

print("Q2 RDD Result:")
for year, quarter, avg_rev in highest_avg_rdd:
    print(f"Year: {year}, Quarter: {quarter}, Avg Revenue Per Seat: {avg_rev:.2f}")

### Q3. Calculate the total number of booked seats for each year and quarter

In [7]:
# Q3: Total number of booked seats for each year and quarter
total_booked_seats = df.groupBy("Year", "Quarter") \
    .agg(sum("booked_seats").alias("Total_Booked_Seats")) \
    .orderBy("Year", "Quarter")

total_booked_seats.show()

+----+-------+------------------+
|Year|Quarter|Total_Booked_Seats|
+----+-------+------------------+
|1995|      1|             46561|
|1995|      2|             37443|
|1995|      3|             34128|
|1995|      4|             30388|
|1996|      1|             47808|
|1996|      2|             43020|
|1996|      3|             38952|
|1996|      4|             37443|
|1997|      1|             35067|
|1997|      2|             46565|
|1997|      3|             38886|
|1997|      4|             37454|
|1998|      1|             31315|
|1998|      2|             30852|
|1998|      3|             38118|
|1998|      4|             35393|
|1999|      1|             47453|
|1999|      2|             38243|
|1999|      3|             33048|
|1999|      4|             31256|
+----+-------+------------------+
only showing top 20 rows



#### RDD API Solution

In [ ]:
# Q3 RDD: Total number of booked seats for each year and quarter
total_booked_rdd = rdd.map(lambda row: ((row.Year, row.Quarter), row.booked_seats)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortByKey()

print("Q3 RDD Result:")
for key, total_seats in total_booked_rdd.collect():
    print(f"Year: {key[0]}, Quarter: {key[1]}, Total Booked Seats: {total_seats}")

### Q4. Determine the year and quarter with the highest total number of booked seats

In [8]:
# Q4: Year and quarter with highest total number of booked seats
highest_booked_seats = df.groupBy("Year", "Quarter") \
    .agg(sum("booked_seats").alias("Total_Booked_Seats")) \
    .orderBy(desc("Total_Booked_Seats")) \
    .limit(1)

highest_booked_seats.show()

+----+-------+------------------+
|Year|Quarter|Total_Booked_Seats|
+----+-------+------------------+
|2010|      1|             49678|
+----+-------+------------------+



#### RDD API Solution

In [ ]:
# Q4 RDD: Year and quarter with highest total number of booked seats
highest_booked_rdd = rdd.map(lambda row: ((row.Year, row.Quarter), row.booked_seats)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(1)

print("Q4 RDD Result:")
for key, total_seats in highest_booked_rdd:
    print(f"Year: {key[0]}, Quarter: {key[1]}, Total Booked Seats: {total_seats}")

### Q5. Calculate the total revenue generated for each year and quarter

In [9]:
# Q5: Total revenue for each year and quarter (in millions)
# Revenue = Avg_rev_per_seat * booked_seats
from pyspark.sql.functions import round as spark_round

df_with_revenue = df.withColumn("Total_Revenue", col("Avg_rev_per_seat") * col("booked_seats"))

total_revenue_per_year_quarter = df_with_revenue.groupBy("Year", "Quarter") \
    .agg(spark_round(sum("Total_Revenue") / 1000000, 2).alias("Total_Revenue_Millions")) \
    .orderBy("Year", "Quarter")

total_revenue_per_year_quarter.show()

+----+-------+----------------------+
|Year|Quarter|Total_Revenue_Millions|
+----+-------+----------------------+
|1995|      1|                 13.82|
|1995|      2|                 11.11|
|1995|      3|                  9.81|
|1995|      4|                  8.75|
|1996|      1|                 13.58|
|1996|      2|                 11.86|
|1996|      3|                  10.5|
|1996|      4|                 10.42|
|1997|      1|                  9.94|
|1997|      2|                 13.48|
|1997|      3|                 10.98|
|1997|      4|                 10.99|
|1998|      1|                  9.54|
|1998|      2|                  9.29|
|1998|      3|                 12.02|
|1998|      4|                 11.19|
|1999|      1|                 15.74|
|1999|      2|                 12.59|
|1999|      3|                 10.48|
|1999|      4|                  9.94|
+----+-------+----------------------+
only showing top 20 rows



#### RDD API Solution

In [ ]:
# Q5 RDD: Total revenue for each year and quarter (in millions)
# Revenue = Avg_rev_per_seat * booked_seats
total_revenue_rdd = rdd.map(lambda row: ((row.Year, row.Quarter), row.Avg_rev_per_seat * row.booked_seats)) \
    .reduceByKey(lambda a, b: a + b) \
    .mapValues(lambda x: round(x / 1000000, 2)) \
    .sortByKey()

print("Q5 RDD Result:")
for key, total_rev in total_revenue_rdd.collect():
    print(f"Year: {key[0]}, Quarter: {key[1]}, Total Revenue (Millions): ${total_rev}M")

### Q6. Identify the year and quarter with the highest total revenue

In [10]:
# Q6: Year and quarter with highest total revenue (in millions)
highest_revenue = df_with_revenue.groupBy("Year", "Quarter") \
    .agg(spark_round(sum("Total_Revenue") / 1000000, 2).alias("Total_Revenue_Millions")) \
    .orderBy(desc("Total_Revenue_Millions")) \
    .limit(1)

highest_revenue.show()

+----+-------+----------------------+
|Year|Quarter|Total_Revenue_Millions|
+----+-------+----------------------+
|2014|      4|                 18.82|
+----+-------+----------------------+



#### RDD API Solution

In [ ]:
# Q6 RDD: Year and quarter with highest total revenue (in millions)
highest_revenue_rdd = rdd.map(lambda row: ((row.Year, row.Quarter), row.Avg_rev_per_seat * row.booked_seats)) \
    .reduceByKey(lambda a, b: a + b) \
    .mapValues(lambda x: round(x / 1000000, 2)) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(1)

print("Q6 RDD Result:")
for key, total_rev in highest_revenue_rdd:
    print(f"Year: {key[0]}, Quarter: {key[1]}, Total Revenue (Millions): ${total_rev}M")

### Q7. Find the average revenue per seat across different years

In [11]:
# Q7: Average revenue per seat across different years
avg_revenue_per_year = df.groupBy("Year") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy("Year")

avg_revenue_per_year.show()

+----+--------------------+
|Year|Avg_Revenue_Per_Seat|
+----+--------------------+
|1995|            292.2475|
|1996|            276.8925|
|1997|             287.155|
|1998|             309.285|
|1999|            324.0575|
|2000|            339.0325|
|2001|            319.7975|
|2002|             312.525|
|2003|            315.4675|
|2004|             305.875|
|2005|             307.185|
|2006|               328.3|
|2007|              325.14|
|2008|            346.1575|
|2009|              310.61|
|2010|            335.8325|
|2011|  363.63250000000005|
|2012|             374.675|
|2013|            382.0025|
|2014|               391.7|
+----+--------------------+
only showing top 20 rows



#### RDD API Solution

In [ ]:
# Q7 RDD: Average revenue per seat across different years
avg_revenue_by_year_rdd = rdd.map(lambda row: (row.Year, row.Avg_rev_per_seat)) \
    .aggregateByKey((0.0, 0), 
                    lambda acc, value: (acc[0] + value, acc[1] + 1),
                    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2)) \
    .sortByKey()

print("Q7 RDD Result:")
for year, avg_rev in avg_revenue_by_year_rdd.collect():
    print(f"Year: {year}, Avg Revenue Per Seat: {avg_rev:.2f}")

### Q8. Determine the year with the highest average revenue per seat

In [12]:
# Q8: Year with highest average revenue per seat
highest_avg_revenue_year = df.groupBy("Year") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy(desc("Avg_Revenue_Per_Seat")) \
    .limit(1)

highest_avg_revenue_year.show()

+----+--------------------+
|Year|Avg_Revenue_Per_Seat|
+----+--------------------+
|2014|               391.7|
+----+--------------------+



#### RDD API Solution

In [ ]:
# Q8 RDD: Year with highest average revenue per seat
highest_avg_year_rdd = rdd.map(lambda row: (row.Year, row.Avg_rev_per_seat)) \
    .aggregateByKey((0.0, 0), 
                    lambda acc, value: (acc[0] + value, acc[1] + 1),
                    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2)) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(1)

print("Q8 RDD Result:")
for year, avg_rev in highest_avg_year_rdd:
    print(f"Year: {year}, Avg Revenue Per Seat: {avg_rev:.2f}")

### Q9. Calculate the overall average revenue per seat for the entire dataset

In [13]:
# Q9: Overall average revenue per seat for entire dataset
overall_avg_revenue = df.agg(avg("Avg_rev_per_seat").alias("Overall_Avg_Revenue_Per_Seat"))
overall_avg_revenue.show()

+----------------------------+
|Overall_Avg_Revenue_Per_Seat|
+----------------------------+
|          329.74750000000006|
+----------------------------+



#### RDD API Solution

In [ ]:
# Q9 RDD: Overall average revenue per seat for entire dataset
# Calculate sum and count, then compute average
total_and_count = rdd.map(lambda row: (row.Avg_rev_per_seat, 1)) \
    .reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))

overall_avg = round(total_and_count[0] / total_and_count[1], 2)

print("Q9 RDD Result:")
print(f"Overall Average Revenue Per Seat: {overall_avg:.2f}")

---
# Part 2: Spark SQL (Questions 10-19)

### Q10. Create a database and table to store the airline CSV data

In [14]:
# Q10: Create database and table
spark.sql("CREATE DATABASE IF NOT EXISTS airlines_db")
spark.sql("USE airlines_db")

# Register DataFrame as a temporary view/table
df.createOrReplaceTempView("airlines")

# Verify table creation
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM airlines LIMIT 5").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         | airlines|       true|
+---------+---------+-----------+

+----+-------+----------------+------------+
|Year|Quarter|Avg_rev_per_seat|booked_seats|
+----+-------+----------------+------------+
|1995|      1|           296.9|       46561|
|1995|      2|           296.8|       37443|
|1995|      3|          287.51|       34128|
|1995|      4|          287.78|       30388|
|1996|      1|          283.97|       47808|
+----+-------+----------------+------------+



### Q11. Calculate the average revenue per seat for each year and quarter using SQL

In [15]:
# Q11: Average revenue per seat for each year and quarter
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

+----+-------+--------------------+
|Year|Quarter|Avg_Revenue_Per_Seat|
+----+-------+--------------------+
|1995|      1|               296.9|
|1995|      2|               296.8|
|1995|      3|              287.51|
|1995|      4|              287.78|
|1996|      1|              283.97|
|1996|      2|              275.78|
|1996|      3|              269.49|
|1996|      4|              278.33|
|1997|      1|               283.4|
|1997|      2|              289.44|
|1997|      3|              282.27|
|1997|      4|              293.51|
|1998|      1|              304.74|
|1998|      2|              300.97|
|1998|      3|              315.25|
|1998|      4|              316.18|
|1999|      1|              331.74|
|1999|      2|              329.34|
|1999|      3|              317.22|
|1999|      4|              317.93|
+----+-------+--------------------+
only showing top 20 rows



### Q12. Find the year and quarter with the highest average revenue per seat using SQL

In [16]:
# Q12: Year and quarter with highest average revenue per seat
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
spark.sql(query).show()

+----+-------+--------------------+
|Year|Quarter|Avg_Revenue_Per_Seat|
+----+-------+--------------------+
|2014|      3|              396.37|
+----+-------+--------------------+



### Q13. Calculate the total number of booked seats for each year and quarter using SQL

In [17]:
# Q13: Total number of booked seats for each year and quarter
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

+----+-------+------------------+
|Year|Quarter|Total_Booked_Seats|
+----+-------+------------------+
|1995|      1|             46561|
|1995|      2|             37443|
|1995|      3|             34128|
|1995|      4|             30388|
|1996|      1|             47808|
|1996|      2|             43020|
|1996|      3|             38952|
|1996|      4|             37443|
|1997|      1|             35067|
|1997|      2|             46565|
|1997|      3|             38886|
|1997|      4|             37454|
|1998|      1|             31315|
|1998|      2|             30852|
|1998|      3|             38118|
|1998|      4|             35393|
|1999|      1|             47453|
|1999|      2|             38243|
|1999|      3|             33048|
|1999|      4|             31256|
+----+-------+------------------+
only showing top 20 rows



### Q14. Determine the year and quarter with the highest total number of booked seats using SQL

In [18]:
# Q14: Year and quarter with highest total number of booked seats
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Booked_Seats DESC
LIMIT 1
"""
spark.sql(query).show()

+----+-------+------------------+
|Year|Quarter|Total_Booked_Seats|
+----+-------+------------------+
|2010|      1|             49678|
+----+-------+------------------+



### Q15. Calculate the total revenue generated for each year and quarter using SQL

In [19]:
# Q15: Total revenue for each year and quarter (in millions)
query = """
SELECT Year, Quarter,
       ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) AS Total_Revenue_Millions
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

+----+-------+----------------------+
|Year|Quarter|Total_Revenue_Millions|
+----+-------+----------------------+
|1995|      1|                 13.82|
|1995|      2|                 11.11|
|1995|      3|                  9.81|
|1995|      4|                  8.75|
|1996|      1|                 13.58|
|1996|      2|                 11.86|
|1996|      3|                  10.5|
|1996|      4|                 10.42|
|1997|      1|                  9.94|
|1997|      2|                 13.48|
|1997|      3|                 10.98|
|1997|      4|                 10.99|
|1998|      1|                  9.54|
|1998|      2|                  9.29|
|1998|      3|                 12.02|
|1998|      4|                 11.19|
|1999|      1|                 15.74|
|1999|      2|                 12.59|
|1999|      3|                 10.48|
|1999|      4|                  9.94|
+----+-------+----------------------+
only showing top 20 rows



### Q16. Identify the year and quarter with the highest total revenue using SQL

In [20]:
# Q16: Year and quarter with highest total revenue (in millions)
query = """
SELECT Year, Quarter,
       ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) AS Total_Revenue_Millions
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Revenue_Millions DESC
LIMIT 1
"""
spark.sql(query).show()

+----+-------+----------------------+
|Year|Quarter|Total_Revenue_Millions|
+----+-------+----------------------+
|2014|      4|                 18.82|
+----+-------+----------------------+



### Q17. Find the average revenue per seat across different years using SQL

In [21]:
# Q17: Average revenue per seat across different years
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Year
"""
spark.sql(query).show()

+----+--------------------+
|Year|Avg_Revenue_Per_Seat|
+----+--------------------+
|1995|            292.2475|
|1996|            276.8925|
|1997|             287.155|
|1998|             309.285|
|1999|            324.0575|
|2000|            339.0325|
|2001|            319.7975|
|2002|             312.525|
|2003|            315.4675|
|2004|             305.875|
|2005|             307.185|
|2006|               328.3|
|2007|              325.14|
|2008|            346.1575|
|2009|              310.61|
|2010|            335.8325|
|2011|  363.63250000000005|
|2012|             374.675|
|2013|            382.0025|
|2014|               391.7|
+----+--------------------+
only showing top 20 rows



### Q18. Determine the year with the highest average revenue per seat using SQL

In [22]:
# Q18: Year with highest average revenue per seat
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
spark.sql(query).show()

+----+--------------------+
|Year|Avg_Revenue_Per_Seat|
+----+--------------------+
|2014|               391.7|
+----+--------------------+



### Q19. Calculate the overall average revenue per seat for the entire dataset using SQL

In [23]:
# Q19: Overall average revenue per seat for entire dataset
query = """
SELECT AVG(Avg_rev_per_seat) AS Overall_Avg_Revenue_Per_Seat
FROM airlines
"""
spark.sql(query).show()

+----------------------------+
|Overall_Avg_Revenue_Per_Seat|
+----------------------------+
|          329.74750000000006|
+----------------------------+



---
# Part 3: Data Analysis Questions (Questions 20-39)

### Q20-22: Average revenue per seat analysis by year and quarter

In [24]:
# Q20-22: Combined analysis
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q20 = spark.sql(query)
result_q20.show(50)

+----+-------+--------------------+
|Year|Quarter|Avg_Revenue_Per_Seat|
+----+-------+--------------------+
|1995|      1|               296.9|
|1995|      2|               296.8|
|1995|      3|              287.51|
|1995|      4|              287.78|
|1996|      1|              283.97|
|1996|      2|              275.78|
|1996|      3|              269.49|
|1996|      4|              278.33|
|1997|      1|               283.4|
|1997|      2|              289.44|
|1997|      3|              282.27|
|1997|      4|              293.51|
|1998|      1|              304.74|
|1998|      2|              300.97|
|1998|      3|              315.25|
|1998|      4|              316.18|
|1999|      1|              331.74|
|1999|      2|              329.34|
|1999|      3|              317.22|
|1999|      4|              317.93|
|2000|      1|              340.23|
|2000|      2|              339.16|
|2000|      3|              336.66|
|2000|      4|              340.08|
|2001|      1|              

### Q23-25: Total booked seats analysis by year and quarter

In [25]:
# Q23-25: Combined analysis
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q23 = spark.sql(query)
result_q23.show(50)

+----+-------+------------------+
|Year|Quarter|Total_Booked_Seats|
+----+-------+------------------+
|1995|      1|             46561|
|1995|      2|             37443|
|1995|      3|             34128|
|1995|      4|             30388|
|1996|      1|             47808|
|1996|      2|             43020|
|1996|      3|             38952|
|1996|      4|             37443|
|1997|      1|             35067|
|1997|      2|             46565|
|1997|      3|             38886|
|1997|      4|             37454|
|1998|      1|             31315|
|1998|      2|             30852|
|1998|      3|             38118|
|1998|      4|             35393|
|1999|      1|             47453|
|1999|      2|             38243|
|1999|      3|             33048|
|1999|      4|             31256|
|2000|      1|             48159|
|2000|      2|             38329|
|2000|      3|             37785|
|2000|      4|             30103|
|2001|      1|             43853|
|2001|      2|             43048|
|2001|      3|

### Q26-29: Total revenue analysis by year and quarter

In [26]:
# Q26-29: Revenue trends (in millions)
query = """
SELECT Year,
       ROUND(AVG(Avg_rev_per_seat), 2) AS Avg_Revenue_Per_Seat,
       SUM(booked_seats) AS Total_Booked_Seats,
       ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) AS Total_Revenue_Millions
FROM airlines
GROUP BY Year
ORDER BY Year
"""
result_q29 = spark.sql(query)
print("\n--- Trends and Patterns Over Years ---")
result_q29.show(50)


--- Trends and Patterns Over Years ---
+----+--------------------+------------------+----------------------+
|Year|Avg_Revenue_Per_Seat|Total_Booked_Seats|Total_Revenue_Millions|
+----+--------------------+------------------+----------------------+
|1995|              292.25|            148520|                 43.49|
|1996|              276.89|            167223|                 46.36|
|1997|              287.16|            157972|                 45.39|
|1998|              309.29|            135678|                 42.04|
|1999|              324.06|            150000|                 48.76|
|2000|              339.03|            154376|                 52.34|
|2001|               319.8|            173598|                 55.53|
|2002|              312.53|            152195|                  47.5|
|2003|              315.47|            156153|                 49.27|
|2004|              305.88|            164800|                 50.63|
|2005|              307.19|            150610|    

---
## Summary and Insights

This notebook demonstrates:
1. **DataFrame API**: Questions 1-9 using PySpark DataFrame operations
2. **Spark SQL**: Questions 10-19 using SQL queries
3. **Data Analysis**: Questions 20-39 providing comprehensive insights

### Key Findings:
- Average revenue per seat trends by year and quarter
- Total booked seats patterns
- Total revenue calculations (displayed in millions)
- Seasonal trends identification
- Year-over-year comparisons